# Run Results Diagnostics

This notebook analyzes a single simulation run using the artifacts that are actually present in this log directory.

It is designed to be robust when the exchange log only contains final summary events. In that case, the notebook falls back to end-of-run diagnostics from `summary_log.bz2`, per-agent summary logs, and the fundamental oracle log.

What this notebook does well:
- summarizes exchange-level activity and rejection reasons
- builds a per-agent results table from `summary_log.bz2`
- highlights bankrupt / underwater agents and suspicious states
- shows fundamental oracle behavior for the run

What it cannot recover from the current logs:
- final positions per agent unless they were logged explicitly
- fill-by-fill or mark-price time series if the exchange did not log them

This root notebook auto-detects the active run directory and prefers `log/latest_run.json` when available. Otherwise it falls back to the most recently updated run directory under `log/`.


In [ ]:
from pathlib import Path
import ast
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def is_run_dir(path: Path):
    return (path / 'summary_log.bz2').exists() and (path / 'PERP_EXCHANGE.bz2').exists()

def latest_run_dir():
    latest_manifest = Path('log/latest_run.json')
    if latest_manifest.exists():
        try:
            payload = json.loads(latest_manifest.read_text(encoding='utf-8'))
            manifest_dir = Path(payload.get('log_path', ''))
            if manifest_dir.exists() and is_run_dir(manifest_dir):
                return manifest_dir
        except Exception:
            pass
    candidates = []
    here = Path('.').resolve()
    if is_run_dir(here):
        candidates.append(here)
    log_root = Path('log')
    if log_root.exists():
        candidates.extend(
            sorted(
                [path.resolve() for path in log_root.iterdir() if path.is_dir() and is_run_dir(path)],
                key=lambda path: (path / 'summary_log.bz2').stat().st_mtime,
                reverse=True,
            )
        )
    return candidates[0] if candidates else Path('.').resolve()

RUN_DIR = latest_run_dir()
SUMMARY_PATH = RUN_DIR / 'summary_log.bz2'
EXCHANGE_PATH = RUN_DIR / 'PERP_EXCHANGE.bz2'
DEPLOYER_PATH = RUN_DIR / 'ORACLE_DEPLOYER.bz2'
FUNDAMENTAL_PATHS = sorted(RUN_DIR.glob('fundamental_*.bz2'))
SPECIAL_STEMS = {'summary_log', 'PERP_EXCHANGE', 'ORACLE_DEPLOYER'}

def load_bz2(path: Path):
    if not path.exists():
        return None
    return pd.read_pickle(path, compression='bz2')

def parse_event(value):
    if isinstance(value, str):
        stripped = value.strip()
        if stripped.startswith('{') or stripped.startswith('['):
            try:
                return ast.literal_eval(stripped)
            except Exception:
                return value
    return value

def nested_value(payload, *keys, default=0):
    cur = payload
    for key in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key, default)
    return cur

summary = load_bz2(SUMMARY_PATH)
exchange_log = load_bz2(EXCHANGE_PATH)
deployer_log = load_bz2(DEPLOYER_PATH)
fundamentals = {path.stem.replace('fundamental_', ''): load_bz2(path) for path in FUNDAMENTAL_PATHS}

print(f'Run directory: {RUN_DIR.resolve()}')
print(f'Summary log rows: {0 if summary is None else len(summary)}')
print(f'Exchange log rows: {0 if exchange_log is None else len(exchange_log)}')
print(f'Deployer log rows: {0 if deployer_log is None else len(deployer_log)}')
print(f'Fundamental logs: {list(fundamentals)}')


In [ ]:
assert summary is not None, 'summary_log.bz2 not found'

summary = summary.copy()
summary['ParsedEvent'] = summary['Event'].apply(parse_event)

exchange_activity = summary.loc[summary['EventType'] == 'PERP_EXCHANGE_ACTIVITY_SUMMARY', 'ParsedEvent']
exchange_rejections = summary.loc[summary['EventType'] == 'PERP_EXCHANGE_REJECTION_REASONS', 'ParsedEvent']
exchange_open_orders = summary.loc[summary['EventType'] == 'PERP_EXCHANGE_OPEN_ORDERS_AT_STOP', 'ParsedEvent']

exchange_activity = exchange_activity.iloc[0] if not exchange_activity.empty else {}
exchange_rejections = exchange_rejections.iloc[0] if not exchange_rejections.empty else {}
exchange_open_orders = exchange_open_orders.iloc[0] if not exchange_open_orders.empty else {}

print('Exchange log event types present:')
if exchange_log is not None and len(exchange_log) > 0:
    print(exchange_log['EventType'].value_counts().to_string())
else:
    print('  <none>')

rich_exchange_events = ['SET_ORACLE', 'MARK_PRICE_UPDATE', 'FUNDING_SETTLED', 'QUERY_MARK_PRICE']
available_exchange_events = set(exchange_log['EventType']) if exchange_log is not None else set()
missing_exchange_events = [name for name in rich_exchange_events if name not in available_exchange_events]
if missing_exchange_events:
    print()
    print('Detailed exchange time-series events are not available for this run:')
    print('  ' + ', '.join(missing_exchange_events))
    print('Notebook will use end-of-run summaries instead of pretending those series exist.')

exchange_activity_df = pd.DataFrame.from_dict(exchange_activity.get('global', {}), orient='index', columns=['count']).rename_axis('metric')
exchange_rejection_df = pd.DataFrame.from_dict(exchange_rejections.get('global', {}), orient='index', columns=['count']).rename_axis('rejection_reason')
if exchange_open_orders:
    exchange_open_df = pd.DataFrame(
        [
            {
                'symbol': symbol,
                **details,
            }
            for symbol, details in exchange_open_orders.items()
        ]
    ).set_index('symbol')
else:
    exchange_open_df = pd.DataFrame(columns=['resting', 'trigger', 'dormant_trigger'])

display(exchange_activity_df)
display(exchange_rejection_df)
display(exchange_open_df)


In [ ]:
agent_rows = []
for agent_id, group in summary.groupby('AgentID'):
    strategy = group['AgentStrategy'].dropna().iloc[0]
    if strategy == 'PerpExchangeAgent':
        continue
    row = {'AgentID': int(agent_id), 'AgentStrategy': strategy}
    for event_type in [
        'STARTING_CASH', 'FINAL_BALANCE', 'ENDING_EQUITY', 'FINAL_VALUATION',
        'PERP_ACTIVITY_SUMMARY', 'PERP_REJECTION_REASONS', 'PERP_OPEN_ORDERS_AT_STOP'
    ]:
        values = group.loc[group['EventType'] == event_type, 'ParsedEvent']
        if not values.empty:
            row[event_type] = values.iloc[0]
    agent_rows.append(row)

agents = pd.DataFrame(agent_rows).sort_values('AgentID').reset_index(drop=True)

agent_log_stems = []
for path in RUN_DIR.glob('*.bz2'):
    stem = path.stem
    if stem in SPECIAL_STEMS or stem.startswith('fundamental_'):
        continue
    agent_log_stems.append(stem)

prefix_order = {'Noise': 0, 'Momentum': 1, 'Value': 2, 'Fundamentalist': 3, 'Chartist': 4}

def stem_sort_key(stem):
    match = re.match(r'([A-Za-z]+)_(\d+)$', stem)
    if match:
        prefix, idx = match.groups()
        return (prefix_order.get(prefix, 999), prefix, int(idx))
    return (998, stem, -1)

ordered_stems = sorted(agent_log_stems, key=stem_sort_key)
if len(ordered_stems) == len(agents):
    agents['AgentName'] = pd.Series(ordered_stems, index=agents.index)
else:
    agents['AgentName'] = agents['AgentStrategy'] + '_' + agents.groupby('AgentStrategy').cumcount().astype(str)
    print('Warning: agent log count did not match summary agent count, so names were synthesized.')

for col in ['STARTING_CASH', 'FINAL_BALANCE', 'ENDING_EQUITY', 'FINAL_VALUATION']:
    if col in agents.columns:
        agents[col] = pd.to_numeric(agents[col], errors='coerce')

agents['pnl'] = agents['ENDING_EQUITY'] - agents['STARTING_CASH']
agents['pnl_pct'] = agents['pnl'] / agents['STARTING_CASH']
agents['cash_drift'] = agents['FINAL_BALANCE'] - agents['STARTING_CASH']
agents['is_negative_equity'] = agents['ENDING_EQUITY'] < 0
agents['is_negative_balance'] = agents['FINAL_BALANCE'] < 0

def symbol_payload(payload):
    if isinstance(payload, dict) and payload:
        symbol = next(iter(payload))
        return symbol, payload[symbol]
    return None, {}

activity_payloads = []
for payload in agents.get('PERP_ACTIVITY_SUMMARY', []):
    symbol, info = symbol_payload(payload)
    activity_payloads.append({
        'symbol': symbol,
        'submitted': nested_value(info, 'submitted'),
        'accepted': nested_value(info, 'accepted'),
        'rejected': nested_value(info, 'rejected'),
        'executed': nested_value(info, 'executed'),
        'cancelled': nested_value(info, 'cancelled'),
        'open_orders_at_stop': nested_value(info, 'open_orders_at_stop'),
        'status': nested_value(info, 'status', default=''),
        'rej_OI_CAP': nested_value(info, 'rejection_reasons', 'OI_CAP'),
        'rej_UNFUNDED_ACCOUNT': nested_value(info, 'rejection_reasons', 'UNFUNDED_ACCOUNT'),
        'rej_REDUCE_ONLY_NO_POSITION': nested_value(info, 'rejection_reasons', 'REDUCE_ONLY_NO_POSITION'),
        'rej_MAX_ORDER_VALUE': nested_value(info, 'rejection_reasons', 'MAX_ORDER_VALUE'),
        'rej_MIN_ORDER_VALUE': nested_value(info, 'rejection_reasons', 'MIN_ORDER_VALUE'),
    })

activity_df = pd.DataFrame(activity_payloads)
agents = pd.concat([agents, activity_df], axis=1)
agents['accept_rate'] = np.where(agents['submitted'] > 0, agents['accepted'] / agents['submitted'], np.nan)
agents['exec_per_accept'] = np.where(agents['accepted'] > 0, agents['executed'] / agents['accepted'], np.nan)
agents['bankrupt_flat'] = (agents['ENDING_EQUITY'] < 0) & (agents['status'] == 'filled')
agents['status_open_mismatch'] = (agents['open_orders_at_stop'] > 0) & (agents['status'] == 'filled')

key_cols = [
    'AgentID', 'AgentName', 'AgentStrategy', 'STARTING_CASH', 'FINAL_BALANCE', 'ENDING_EQUITY',
    'pnl', 'pnl_pct', 'submitted', 'accepted', 'rejected', 'executed', 'cancelled',
    'open_orders_at_stop', 'status', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP',
    'rej_REDUCE_ONLY_NO_POSITION', 'rej_MAX_ORDER_VALUE', 'rej_MIN_ORDER_VALUE',
    'accept_rate', 'exec_per_accept', 'bankrupt_flat', 'status_open_mismatch'
]

display(agents[key_cols].sort_values(['AgentStrategy', 'AgentID']).reset_index(drop=True))


In [ ]:
strategy_summary = agents.groupby('AgentStrategy').agg(
    agents=('AgentID', 'count'),
    negative_equity=('is_negative_equity', 'sum'),
    negative_balance=('is_negative_balance', 'sum'),
    bankrupt_flat=('bankrupt_flat', 'sum'),
    status_open_mismatch=('status_open_mismatch', 'sum'),
    mean_equity=('ENDING_EQUITY', 'mean'),
    median_equity=('ENDING_EQUITY', 'median'),
    min_equity=('ENDING_EQUITY', 'min'),
    max_equity=('ENDING_EQUITY', 'max'),
    mean_pnl=('pnl', 'mean'),
    median_pnl=('pnl', 'median'),
    mean_submitted=('submitted', 'mean'),
    mean_accepted=('accepted', 'mean'),
    mean_rejected=('rejected', 'mean'),
    mean_executed=('executed', 'mean'),
    mean_unfunded_rejections=('rej_UNFUNDED_ACCOUNT', 'mean'),
    mean_oi_cap_rejections=('rej_OI_CAP', 'mean'),
).sort_index()

display(strategy_summary)

print('Worst ending equity:')
display(agents.sort_values('ENDING_EQUITY').head(20)[['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'FINAL_BALANCE', 'status', 'open_orders_at_stop', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP']])

print('Best ending equity:')
display(agents.sort_values('ENDING_EQUITY', ascending=False).head(20)[['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'FINAL_BALANCE', 'status', 'open_orders_at_stop', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP']])

print('Agents with negative equity:')
display(agents.loc[agents['is_negative_equity'], ['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'FINAL_BALANCE', 'status', 'open_orders_at_stop', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'rej_REDUCE_ONLY_NO_POSITION']].sort_values('ENDING_EQUITY'))

print('Suspicious summary states: status says filled but open orders remain at stop')
display(agents.loc[agents['status_open_mismatch'], ['AgentID', 'AgentName', 'AgentStrategy', 'status', 'open_orders_at_stop', 'executed', 'ENDING_EQUITY']].sort_values(['AgentStrategy', 'AgentID']))


In [ ]:
agents_plot = agents.copy()
strategy_labels = {
    'PerpNoiseAgent': 'Noise',
    'ChiarellaAgent': 'Chiarella',
}
agents_plot['StrategyLabel'] = agents_plot['AgentStrategy'].map(strategy_labels).fillna(agents_plot['AgentStrategy'])
agents_plot['rejection_rate'] = np.where(agents_plot['submitted'] > 0, agents_plot['rejected'] / agents_plot['submitted'], np.nan)
agents_plot['unfunded_share_of_rejections'] = np.where(agents_plot['rejected'] > 0, agents_plot['rej_UNFUNDED_ACCOUNT'] / agents_plot['rejected'], 0.0)
agents_plot['equity_multiple'] = np.where(agents_plot['STARTING_CASH'] > 0, agents_plot['ENDING_EQUITY'] / agents_plot['STARTING_CASH'], np.nan)

plot_groups = [(label, group.copy()) for label, group in agents_plot.groupby('StrategyLabel')]
colors = {'Noise': '#d55e00', 'Chiarella': '#0072b2'}

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

box_data = [group['ENDING_EQUITY'].dropna().values for label, group in plot_groups]
box_labels = [label for label, _ in plot_groups]
box = axes[0, 0].boxplot(box_data, labels=box_labels, patch_artist=True)
for patch, label in zip(box['boxes'], box_labels):
    patch.set_facecolor(colors.get(label, '#999999'))
    patch.set_alpha(0.5)
axes[0, 0].axhline(0, color='black', linewidth=0.8)
axes[0, 0].set_title('Ending Equity By Strategy')
axes[0, 0].set_ylabel('Ending equity')
axes[0, 0].grid(True, alpha=0.3)

for label, group in plot_groups:
    axes[0, 1].scatter(group['rej_UNFUNDED_ACCOUNT'], group['ENDING_EQUITY'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[0, 1].axhline(0, color='black', linewidth=0.8)
axes[0, 1].set_title('UNFUNDED_ACCOUNT Rejections vs Ending Equity')
axes[0, 1].set_xlabel('UNFUNDED_ACCOUNT rejections')
axes[0, 1].set_ylabel('Ending equity')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

for label, group in plot_groups:
    axes[1, 0].scatter(group['accept_rate'], group['pnl'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].set_title('Acceptance Rate vs PnL')
axes[1, 0].set_xlabel('accept rate')
axes[1, 0].set_ylabel('PnL')
axes[1, 0].grid(True, alpha=0.3)

for label, group in plot_groups:
    axes[1, 1].scatter(group['rejection_rate'], group['unfunded_share_of_rejections'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[1, 1].set_title('Rejection Rate vs Share Of Rejections That Are Unfunded')
axes[1, 1].set_xlabel('rejection rate')
axes[1, 1].set_ylabel('UNFUNDED share of rejections')
axes[1, 1].set_ylim(-0.05, 1.05)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Agents with the most UNFUNDED_ACCOUNT rejections:')
display(
    agents_plot.sort_values(['rej_UNFUNDED_ACCOUNT', 'ENDING_EQUITY'], ascending=[False, True]).head(15)[
        ['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'submitted', 'accepted', 'rejected', 'accept_rate', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'status']
    ]
)

print('Agents with the highest rejection rates:')
display(
    agents_plot.sort_values(['rejection_rate', 'rej_UNFUNDED_ACCOUNT'], ascending=[False, False]).head(15)[
        ['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'submitted', 'accepted', 'rejected', 'rejection_rate', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'status']
    ]
)


In [ ]:
# Extra distribution and outlier views
plot_df = agents_plot.copy()
plot_groups = [(label, group.copy()) for label, group in plot_df.groupby('StrategyLabel')]
colors = {'Noise': '#d55e00', 'Chiarella': '#0072b2'}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for label, group in plot_groups:
    axes[0, 0].hist(group['ENDING_EQUITY'].dropna(), bins=20, alpha=0.45, label=label, color=colors.get(label, '#999999'))
axes[0, 0].axvline(0, color='black', linewidth=0.8)
axes[0, 0].set_title('Ending Equity Distribution')
axes[0, 0].set_xlabel('Ending equity')
axes[0, 0].set_ylabel('Agent count')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

neg_counts = plot_df.groupby('StrategyLabel')['is_negative_equity'].sum().sort_values(ascending=False)
axes[0, 1].bar(neg_counts.index, neg_counts.values, color=[colors.get(label, '#999999') for label in neg_counts.index])
axes[0, 1].set_title('Negative-Equity Agents By Strategy')
axes[0, 1].set_ylabel('Count')
axes[0, 1].grid(True, axis='y', alpha=0.3)

worst = plot_df.nsmallest(12, 'ENDING_EQUITY').sort_values('ENDING_EQUITY')
axes[1, 0].barh(worst['AgentName'], worst['ENDING_EQUITY'], color=[colors.get(label, '#999999') for label in worst['StrategyLabel']])
axes[1, 0].axvline(0, color='black', linewidth=0.8)
axes[1, 0].set_title('Worst 12 Agents By Ending Equity')
axes[1, 0].set_xlabel('Ending equity')
axes[1, 0].grid(True, axis='x', alpha=0.3)

best = plot_df.nlargest(12, 'ENDING_EQUITY').sort_values('ENDING_EQUITY')
axes[1, 1].barh(best['AgentName'], best['ENDING_EQUITY'], color=[colors.get(label, '#999999') for label in best['StrategyLabel']])
axes[1, 1].axvline(0, color='black', linewidth=0.8)
axes[1, 1].set_title('Best 12 Agents By Ending Equity')
axes[1, 1].set_xlabel('Ending equity')
axes[1, 1].grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print('Worst-agent table used for the chart:')
display(worst[['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'FINAL_BALANCE', 'submitted', 'accepted', 'rejected', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'status']])

print('Best-agent table used for the chart:')
display(best[['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'FINAL_BALANCE', 'submitted', 'accepted', 'rejected', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'status']])


In [ ]:
# Extra rejection-mix and activity plots
plot_df = agents_plot.copy()
rejection_cols = [
    'rej_UNFUNDED_ACCOUNT',
    'rej_OI_CAP',
    'rej_REDUCE_ONLY_NO_POSITION',
    'rej_MAX_ORDER_VALUE',
    'rej_MIN_ORDER_VALUE',
]
rejection_mix = plot_df.groupby('StrategyLabel')[rejection_cols].sum()
colors = {'Noise': '#d55e00', 'Chiarella': '#0072b2'}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

rejection_mix.plot(kind='bar', stacked=True, ax=axes[0, 0], colormap='tab20c')
axes[0, 0].set_title('Aggregate Rejection Mix By Strategy')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(True, axis='y', alpha=0.3)
axes[0, 0].legend(loc='upper right', fontsize=8)

for label, group in plot_df.groupby('StrategyLabel'):
    axes[0, 1].scatter(group['submitted'], group['ENDING_EQUITY'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[0, 1].axhline(0, color='black', linewidth=0.8)
axes[0, 1].set_title('Submitted Orders vs Ending Equity')
axes[0, 1].set_xlabel('Submitted orders')
axes[0, 1].set_ylabel('Ending equity')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

for label, group in plot_df.groupby('StrategyLabel'):
    axes[1, 0].scatter(group['rej_OI_CAP'], group['ENDING_EQUITY'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].set_title('OI_CAP Rejections vs Ending Equity')
axes[1, 0].set_xlabel('OI_CAP rejections')
axes[1, 0].set_ylabel('Ending equity')
axes[1, 0].grid(True, alpha=0.3)

for label, group in plot_df.groupby('StrategyLabel'):
    axes[1, 1].scatter(group['executed'], group['ENDING_EQUITY'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[1, 1].axhline(0, color='black', linewidth=0.8)
axes[1, 1].set_title('Executions vs Ending Equity')
axes[1, 1].set_xlabel('Executed trades')
axes[1, 1].set_ylabel('Ending equity')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Aggregate rejection mix:')
display(rejection_mix)


In [ ]:
if not fundamentals:
    print('No fundamental logs found.')
else:
    for symbol, fund in fundamentals.items():
        fund = fund.copy()
        fund.index = pd.to_datetime(fund.index)
        print(f'Fundamental summary for {symbol}:')
        display(fund['FundamentalValue'].describe().to_frame('FundamentalValue'))
        print(f'Time range: {fund.index.min()} -> {fund.index.max()}')
        print(f"First value: {fund['FundamentalValue'].iloc[0]:.4f}")
        print(f"Last value:  {fund['FundamentalValue'].iloc[-1]:.4f}")
        print(f"Max value:   {fund['FundamentalValue'].max():.4f}")
        print(f"Min value:   {fund['FundamentalValue'].min():.4f}")
        ax = fund['FundamentalValue'].plot(figsize=(14, 4), title=f'Fundamental oracle series: {symbol}', linewidth=1)
        ax.set_ylabel('Price')
        ax.grid(True, alpha=0.3)
        plt.show()

if exchange_log is not None and len(exchange_log) > 0 and 'MARK_PRICE_UPDATE' in set(exchange_log['EventType']):
    mark = exchange_log.loc[exchange_log['EventType'] == 'MARK_PRICE_UPDATE'].copy().reset_index()
    parsed = mark['Event'].str.split(',', expand=True)
    if parsed.shape[1] >= 3:
        mark['symbol'] = parsed[0]
        mark['mark_price'] = parsed[1].astype(float)
        mark['oracle_price'] = parsed[2].astype(float)
        mark['EventTime'] = pd.to_datetime(mark['EventTime'])
        mark['premium_bps'] = (mark['mark_price'] - mark['oracle_price']) / mark['oracle_price'] * 10000
        display(mark[['EventTime', 'symbol', 'mark_price', 'oracle_price', 'premium_bps']].head())
        fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
        axes[0].plot(mark['EventTime'], mark['oracle_price'], label='oracle', linewidth=1)
        axes[0].plot(mark['EventTime'], mark['mark_price'], label='mark', linewidth=1)
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        axes[0].set_ylabel('Price')
        axes[1].plot(mark['EventTime'], mark['premium_bps'], linewidth=1)
        axes[1].axhline(0, color='black', linewidth=0.8)
        axes[1].set_ylabel('Premium (bps)')
        axes[1].grid(True, alpha=0.3)
        plt.show()
else:
    print('Detailed mark/oracle exchange events are unavailable for this run, so no mark-price time series can be reconstructed from logs.')


In [ ]:
# --- Order lifecycle accounting ---
# Every order submitted by an agent should be accounted for as accepted or rejected.
# Every accepted order should end as executed, cancelled, or still open at stop.

lifecycle = agents[['AgentName', 'AgentStrategy', 'submitted', 'accepted', 'rejected',
                     'executed', 'cancelled', 'open_orders_at_stop']].copy()

lifecycle['accept_plus_reject'] = lifecycle['accepted'] + lifecycle['rejected']
lifecycle['submit_match'] = lifecycle['submitted'] == lifecycle['accept_plus_reject']

# For executed: each fill touches two sides, so per-agent executed can exceed accepted.
# The invariant is: the exchange global executed should be even (two sides per fill).
global_activity = exchange_activity.get('global', {})
global_executed = global_activity.get('executed', 0)
global_accepted = global_activity.get('accepted', 0)
global_rejected = global_activity.get('rejected', 0)
global_cancelled = global_activity.get('cancelled', 0)

print('=== Order Lifecycle Integrity ===')
print()
mismatches = lifecycle[~lifecycle['submit_match']]
if len(mismatches) == 0:
    print('submitted == accepted + rejected for ALL agents')
else:
    print(f'WARNING: {len(mismatches)} agents have submitted != accepted + rejected:')
    display(mismatches[['AgentName', 'AgentStrategy', 'submitted', 'accepted', 'rejected', 'accept_plus_reject']])

print()
print(f'Global accepted:  {global_accepted:>10,}')
print(f'Global rejected:  {global_rejected:>10,}')
print(f'Global executed:  {global_executed:>10,}')
print(f'Global cancelled: {global_cancelled:>10,}')
print()

agent_total_submitted = lifecycle['submitted'].sum()
agent_total_accepted = lifecycle['accepted'].sum()
agent_total_rejected = lifecycle['rejected'].sum()
print(f'Sum of agent submitted:  {agent_total_submitted:>10,}')
print(f'Sum of agent accepted:   {agent_total_accepted:>10,}')
print(f'Sum of agent rejected:   {agent_total_rejected:>10,}')
print(f'Exchange global accepted: {global_accepted:>10,}')
print(f'Exchange global rejected: {global_rejected:>10,}')
print()
print(f'Agent-vs-exchange accepted match: {agent_total_accepted == global_accepted}')
print(f'Agent-vs-exchange rejected match: {agent_total_rejected == global_rejected}')
print()

is_even_exec = global_executed % 2 == 0
print(f'Global executed is even (two-sided fills): {is_even_exec}  ({global_executed})')
if not is_even_exec:
    print('  NOTE: odd execution count may indicate partial fills or self-trades logged once.')

# Open orders at stop
total_open = lifecycle['open_orders_at_stop'].sum()
exchange_open = exchange_open_orders
print()
print(f'Sum of per-agent open orders at stop: {total_open}')
print(f'Exchange reported open orders at stop:')
display(exchange_open_df)

In [ ]:
# --- Rejection reason coverage ---
# Every rejection should be accounted for by a known reason code.

print('=== Rejection Reason Coverage ===')
print()

agent_rej_cols = ['rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'rej_REDUCE_ONLY_NO_POSITION',
                  'rej_MAX_ORDER_VALUE', 'rej_MIN_ORDER_VALUE']

# Also extract INSUFFICIENT_MARGIN from raw activity payloads
agents['rej_INSUFFICIENT_MARGIN'] = 0
for idx, payload in enumerate(agents.get('PERP_ACTIVITY_SUMMARY', [])):
    _, info = symbol_payload(payload)
    agents.loc[agents.index[idx], 'rej_INSUFFICIENT_MARGIN'] = nested_value(info, 'rejection_reasons', 'INSUFFICIENT_MARGIN')

all_rej_cols = agent_rej_cols + ['rej_INSUFFICIENT_MARGIN']
agents['explained_rejections'] = agents[all_rej_cols].sum(axis=1)
agents['unexplained_rejections'] = agents['rejected'] - agents['explained_rejections']

unexplained = agents[agents['unexplained_rejections'] != 0]
if len(unexplained) == 0:
    print('All per-agent rejections are fully explained by known reason codes.')
else:
    print(f'WARNING: {len(unexplained)} agents have unexplained rejections:')
    display(unexplained[['AgentName', 'AgentStrategy', 'rejected', 'explained_rejections',
                          'unexplained_rejections'] + all_rej_cols])

# Global rejection accounting
global_rej = exchange_rejections.get('global', {})
global_rej_sum = sum(global_rej.values())
print()
print(f'Global rejection reasons sum: {global_rej_sum:,}')
print(f'Global rejected orders:       {global_rejected:,}')
print(f'Global rejection reasons fully account for rejected orders: {global_rej_sum == global_rejected}')
print()
print('Breakdown:')
for reason, count in sorted(global_rej.items(), key=lambda x: -x[1]):
    pct = count / global_rejected * 100 if global_rejected > 0 else 0
    print(f'  {reason:30s} {count:>8,}  ({pct:5.1f}%)')

In [ ]:
# --- Conservation of value (zero-sum check) ---
# In a perpetual futures market with no external inflows, the sum of all PnL
# should equal zero (modulo fees and funding transfers, which are internal).
# A large residual would indicate a platform accounting bug.

print('=== Conservation of Value ===')
print()

total_starting = agents['STARTING_CASH'].sum()
total_ending_equity = agents['ENDING_EQUITY'].sum()
total_ending_balance = agents['FINAL_BALANCE'].sum()
total_pnl = agents['pnl'].sum()

print(f'Total starting cash:    {total_starting:>16,.2f}')
print(f'Total ending equity:    {total_ending_equity:>16,.2f}')
print(f'Total ending balance:   {total_ending_balance:>16,.2f}')
print(f'Net PnL across all agents: {total_pnl:>13,.2f}')
print()

residual_pct = abs(total_pnl) / total_starting * 100 if total_starting > 0 else float('inf')
print(f'Residual as % of total starting cash: {residual_pct:.4f}%')
print()

if residual_pct < 0.01:
    print('Net PnL is near zero â€” value conservation holds.')
elif residual_pct < 1.0:
    print('Small residual â€” likely explained by fees, funding, or liquidation penalties.')
else:
    print('NOTE: Non-trivial residual. This could reflect fees, funding redistribution,')
    print('liquidation insurance fund flows, or (if very large) an accounting issue.')
    print('Investigate fee/funding/liquidation totals to reconcile.')

print()
print('Per-strategy totals:')
strat_totals = agents.groupby('AgentStrategy').agg(
    count=('AgentID', 'count'),
    total_starting=('STARTING_CASH', 'sum'),
    total_equity=('ENDING_EQUITY', 'sum'),
    total_pnl=('pnl', 'sum'),
).sort_index()
display(strat_totals)

In [ ]:
# --- Risk control effectiveness ---
# Did UNFUNDED_ACCOUNT rejections actually gate bankrupt agents?
# Did OI_CAP rejections correlate with position limits being hit?

print('=== Risk Control Effectiveness ===')
print()

# Cluster agents by whether they went negative
solvent = agents[agents['ENDING_EQUITY'] >= 0]
insolvent = agents[agents['ENDING_EQUITY'] < 0]

print(f'Solvent agents:   {len(solvent)}')
print(f'Insolvent agents: {len(insolvent)}')
print()

print('UNFUNDED_ACCOUNT rejections â€” solvent vs insolvent:')
print(f'  Solvent   mean: {solvent["rej_UNFUNDED_ACCOUNT"].mean():8.1f}   '
      f'median: {solvent["rej_UNFUNDED_ACCOUNT"].median():8.1f}   '
      f'max: {solvent["rej_UNFUNDED_ACCOUNT"].max():8.0f}')
if len(insolvent) > 0:
    print(f'  Insolvent mean: {insolvent["rej_UNFUNDED_ACCOUNT"].mean():8.1f}   '
          f'median: {insolvent["rej_UNFUNDED_ACCOUNT"].median():8.1f}   '
          f'max: {insolvent["rej_UNFUNDED_ACCOUNT"].max():8.0f}')
print()

# Agents that went negative but had zero UNFUNDED rejections â€” these were
# solvent during trading but ended negative due to mark-to-market at close
neg_no_unfunded = agents[(agents['ENDING_EQUITY'] < 0) & (agents['rej_UNFUNDED_ACCOUNT'] == 0)]
neg_with_unfunded = agents[(agents['ENDING_EQUITY'] < 0) & (agents['rej_UNFUNDED_ACCOUNT'] > 0)]
print(f'Negative-equity agents WITH unfunded rejections: {len(neg_with_unfunded)}')
print(f'Negative-equity agents WITHOUT unfunded rejections: {len(neg_no_unfunded)}')
if len(neg_no_unfunded) > 0:
    print('  (These agents were solvent during trading but ended underwater at final mark-to-market)')
    display(neg_no_unfunded[['AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'FINAL_BALANCE',
                              'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'accepted', 'executed']])

print()
print('Acceptance rate distribution by solvency:')
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
bins = np.linspace(0, 1, 25)
if len(solvent) > 0:
    ax.hist(solvent['accept_rate'].dropna(), bins=bins, alpha=0.5, label=f'Solvent (n={len(solvent)})', color='#2ca02c')
if len(insolvent) > 0:
    ax.hist(insolvent['accept_rate'].dropna(), bins=bins, alpha=0.5, label=f'Insolvent (n={len(insolvent)})', color='#d62728')
ax.set_xlabel('Acceptance rate')
ax.set_ylabel('Agent count')
ax.set_title('Order Acceptance Rate: Solvent vs Insolvent Agents')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Execution ratio and fill mechanics ---
# In a two-sided market, each match produces two execution events (one per side).
# The ratio of total agent-side executions to global executions reveals whether
# the matching engine is consistent.

print('=== Execution & Fill Mechanics ===')
print()

agent_total_executed = agents['executed'].sum()
print(f'Sum of per-agent executed fills: {agent_total_executed:>12,}')
print(f'Exchange global executed:        {global_executed:>12,}')
print()

if global_executed > 0:
    ratio = agent_total_executed / global_executed
    print(f'Agent-sum / global ratio: {ratio:.4f}')
    if abs(ratio - 1.0) < 0.001:
        print('Ratio ~1.0: exchange counts each side of a fill as one execution globally.')
    elif abs(ratio - 2.0) < 0.01:
        print('Ratio ~2.0: exchange counts fills once globally, agents count their own side.')
    else:
        print(f'Ratio is {ratio:.4f} â€” may reflect partial fills, liquidation fills, or counting conventions.')

print()
print('Executions-per-accepted-order distribution by strategy:')
print('(Values > 1.0 indicate orders that matched multiple times, e.g. resting limit orders)')
exec_ratio_stats = agents.groupby('AgentStrategy')['exec_per_accept'].describe()[['mean', 'std', 'min', '50%', 'max']]
display(exec_ratio_stats)

print()
print('Cancelled orders by strategy:')
cancel_stats = agents.groupby('AgentStrategy')['cancelled'].describe()[['mean', 'std', 'min', '50%', 'max']]
display(cancel_stats)

In [ ]:
# --- Funding & liquidation summary ---
# Platform-level mechanics: how many funding settlements and liquidations occurred,
# and what does that imply about the market's convergence behavior.

print('=== Funding & Liquidation Summary ===')
print()

funding_settlements = global_activity.get('funding_settlements', 0)
liquidations = global_activity.get('liquidations', 0)
adl_events = global_activity.get('adl_events', 0)
trigger_activations = global_activity.get('trigger_activations', 0)

print(f'Funding settlements:    {funding_settlements}')
print(f'Liquidations:           {liquidations}')
print(f'ADL events:             {adl_events}')
print(f'Trigger activations:    {trigger_activations}')
print()

# Estimate sim duration from fundamental log
if fundamentals:
    fund_key = list(fundamentals.keys())[0]
    fund_df = fundamentals[fund_key].copy()
    fund_df.index = pd.to_datetime(fund_df.index)
    sim_hours = (fund_df.index.max() - fund_df.index.min()).total_seconds() / 3600
    print(f'Simulation duration: ~{sim_hours:.1f} hours')
    if funding_settlements > 0:
        hours_per_funding = sim_hours / funding_settlements
        print(f'Average interval between funding settlements: ~{hours_per_funding:.2f} hours')
    if liquidations > 0:
        print(f'Liquidations per hour: ~{liquidations / sim_hours:.1f}')

print()
print('Liquidation context:')
print(f'  Agents with REDUCE_ONLY_NO_POSITION rejections: '
      f'{(agents["rej_REDUCE_ONLY_NO_POSITION"] > 0).sum()}')
print(f'  Total REDUCE_ONLY_NO_POSITION rejections: '
      f'{agents["rej_REDUCE_ONLY_NO_POSITION"].sum()}')
print('  (These indicate the exchange tried to close a position that no longer existed,')
print('   which is expected after a liquidation clears an agent\'s position.)')

print()
if adl_events == 0 and liquidations > 0:
    print('No ADL events despite liquidations â€” the insurance fund / liquidation engine')
    print('handled all underwater positions without resorting to auto-deleveraging.')
elif adl_events > 0:
    print(f'ADL was triggered {adl_events} time(s) â€” liquidation engine could not fully')
    print('close positions at acceptable prices and fell back to auto-deleveraging.')

In [ ]:
# --- Position entry price vs oracle price divergence ---
# Parse final position entry prices from terminal output to measure how far
# the traded price drifted from the oracle. This uses the ENDING_EQUITY and
# FINAL_BALANCE already extracted, plus position info from agent names.

# Extract position entry prices from the agents table where we can infer them.
# entry_price is embedded in the terminal output but not in summary_log.
# Instead, we approximate: if equity != balance, the agent holds a position
# and the difference reflects unrealized PnL at final mark.

agents['unrealized_pnl'] = agents['ENDING_EQUITY'] - agents['FINAL_BALANCE']
agents['has_position'] = agents['unrealized_pnl'].abs() > 0.01

print('=== Position & Unrealized PnL Summary ===')
print()
print(f'Agents carrying positions at end: {agents["has_position"].sum()} / {len(agents)}')
print(f'Agents flat at end:               {(~agents["has_position"]).sum()} / {len(agents)}')
print()

pos_agents = agents[agents['has_position']].copy()
print('Unrealized PnL distribution (agents with positions):')
print(f'  Mean:   {pos_agents["unrealized_pnl"].mean():>12,.2f}')
print(f'  Median: {pos_agents["unrealized_pnl"].median():>12,.2f}')
print(f'  Std:    {pos_agents["unrealized_pnl"].std():>12,.2f}')
print(f'  Min:    {pos_agents["unrealized_pnl"].min():>12,.2f}')
print(f'  Max:    {pos_agents["unrealized_pnl"].max():>12,.2f}')

print()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Unrealized PnL histogram
axes[0].hist(pos_agents['unrealized_pnl'], bins=30, color='#4c72b0', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Unrealized PnL Distribution (Positioned Agents)')
axes[0].set_xlabel('Unrealized PnL (equity - balance)')
axes[0].set_ylabel('Agent count')
axes[0].grid(True, alpha=0.3)

# Balance vs equity scatter â€” points on the diagonal are flat
ax = axes[1]
for strat, group in agents.groupby('AgentStrategy'):
    label = {'PerpNoiseAgent': 'Noise', 'ChiarellaAgent': 'Chiarella'}.get(strat, strat)
    c = {'PerpNoiseAgent': '#d55e00', 'ChiarellaAgent': '#0072b2'}.get(strat, '#999999')
    ax.scatter(group['FINAL_BALANCE'], group['ENDING_EQUITY'], s=35, alpha=0.65, label=label, color=c)
lims = [min(agents['FINAL_BALANCE'].min(), agents['ENDING_EQUITY'].min()),
        max(agents['FINAL_BALANCE'].max(), agents['ENDING_EQUITY'].max())]
ax.plot(lims, lims, 'k--', linewidth=0.8, alpha=0.5, label='balance = equity (flat)')
ax.set_title('Final Balance vs Ending Equity')
ax.set_xlabel('Final balance')
ax.set_ylabel('Ending equity')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Chiarella sub-type breakdown ---
# The ChiarellaAgent config uses sigma_f/sigma_c/sigma_n to create three
# sub-types: Fundamentalist (sigma_f=10), Chartist (sigma_c=10), and
# Chiarella-Noise (sigma_n=10). All are logged as ChiarellaAgent, but
# agent names reveal the sub-type.

chiarella = agents[agents['AgentStrategy'] == 'ChiarellaAgent'].copy()
chiarella['SubType'] = chiarella['AgentName'].str.extract(r'^([A-Za-z]+)_\d+$')[0]
chiarella['SubType'] = chiarella['SubType'].fillna('Unknown')

print('=== Chiarella Sub-Type Breakdown ===')
print()

sub_summary = chiarella.groupby('SubType').agg(
    count=('AgentID', 'count'),
    mean_equity=('ENDING_EQUITY', 'mean'),
    median_equity=('ENDING_EQUITY', 'median'),
    std_equity=('ENDING_EQUITY', 'std'),
    min_equity=('ENDING_EQUITY', 'min'),
    max_equity=('ENDING_EQUITY', 'max'),
    negative_equity=('is_negative_equity', 'sum'),
    mean_pnl=('pnl', 'mean'),
    mean_accept_rate=('accept_rate', 'mean'),
    mean_exec_per_accept=('exec_per_accept', 'mean'),
    mean_rej_unfunded=('rej_UNFUNDED_ACCOUNT', 'mean'),
    mean_rej_oi_cap=('rej_OI_CAP', 'mean'),
    mean_rej_max_val=('rej_MAX_ORDER_VALUE', 'mean'),
)
display(sub_summary)

sub_colors = {'Fundamentalist': '#2ca02c', 'Chartist': '#ff7f0e', 'Noise': '#d62728', 'Unknown': '#999999'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PnL boxplot by sub-type
sub_groups = [(st, g['pnl'].dropna().values) for st, g in chiarella.groupby('SubType')]
box = axes[0].boxplot([v for _, v in sub_groups], labels=[k for k, _ in sub_groups], patch_artist=True)
for patch, (label, _) in zip(box['boxes'], sub_groups):
    patch.set_facecolor(sub_colors.get(label, '#999999'))
    patch.set_alpha(0.5)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('PnL by Chiarella Sub-Type')
axes[0].set_ylabel('PnL')
axes[0].grid(True, alpha=0.3)

# Accept rate by sub-type
for st, group in chiarella.groupby('SubType'):
    axes[1].scatter(group.index, group['accept_rate'], s=35, alpha=0.65,
                    label=st, color=sub_colors.get(st, '#999999'))
axes[1].set_title('Acceptance Rate by Sub-Type')
axes[1].set_xlabel('Agent index')
axes[1].set_ylabel('Acceptance rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Executed trades by sub-type
for st, group in chiarella.groupby('SubType'):
    axes[2].scatter(group['executed'], group['pnl'], s=35, alpha=0.65,
                    label=st, color=sub_colors.get(st, '#999999'))
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_title('Executed Trades vs PnL by Sub-Type')
axes[2].set_xlabel('Executed trades')
axes[2].set_ylabel('PnL')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Platform health scorecard ---
# Summary of all platform integrity checks in one view.

print('=' * 60)
print('  PLATFORM HEALTH SCORECARD')
print('=' * 60)
print()

checks = []

# 1. Order lifecycle
submit_ok = len(mismatches) == 0
checks.append(('Order lifecycle (submitted = accepted + rejected)', submit_ok))

# 2. Agent-exchange totals match
acc_match = agent_total_accepted == global_accepted
rej_match = agent_total_rejected == global_rejected
checks.append(('Agent-exchange accepted totals match', acc_match))
checks.append(('Agent-exchange rejected totals match', rej_match))

# 3. Rejection reasons fully explain rejections
rej_explained = len(unexplained) == 0
checks.append(('All rejections explained by reason codes', rej_explained))

# 4. Global rejection reasons sum to global rejected
rej_sum_ok = global_rej_sum == global_rejected
checks.append(('Global rejection reasons sum = global rejected', rej_sum_ok))

# 5. No ADL (not necessarily a failure, but noteworthy)
no_adl = adl_events == 0
checks.append(('No ADL events (liquidation engine sufficient)', no_adl))

# 6. Funding settlements occurred
funding_ok = funding_settlements > 0
checks.append(('Funding settlements occurred', funding_ok))

# 7. Liquidations occurred when agents went negative
liq_ok = liquidations > 0 if len(insolvent) > 0 else True
checks.append(('Liquidations fired for underwater agents', liq_ok))

# 8. Value conservation (< 1% residual)
conserv_ok = residual_pct < 1.0
checks.append((f'Value conservation (residual < 1% of starting)', conserv_ok))

for label, passed in checks:
    status = 'PASS' if passed else 'WARN'
    marker = '+' if passed else '!'
    print(f'  [{marker}] {status:4s}  {label}')

passed_count = sum(1 for _, p in checks if p)
print()
print(f'{passed_count}/{len(checks)} checks passed.')
if passed_count == len(checks):
    print('All platform integrity checks passed.')
else:
    warn_labels = [label for label, p in checks if not p]
    print(f'Warnings on: {", ".join(warn_labels)}')